# Deep Learning 1 &mdash; Assignment 3

Third assignment for the 2025 Deep Learning course (NWI-IMC070A) of the Radboud University.

-----

**Names:** Nele Haferkorn

**Group:** 13

-----

**Instructions:**
* Fill in your names and the name of your group.
* Answer the questions and complete the code where necessary.
* Keep your answers brief, one or two sentences is usually enough.
* Re-run the whole notebook before you submit your work.
* Save the notebook and submit the `.ipynb` in Brightspace.

## Objectives

This assignment is a continuation of assignment 2. We will work on the same dataset with a similar network architecture.
In this assignment you will

1. Experiment with weight decay
2. Experiment with dropout
4. Experiment with early stopping

In [ ]:
%config InlineBackend.figure_formats = ['png']
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch
import time
import torchvision
import tqdm.notebook as tqdm
import collections
import IPython
import pandas as pd
import copy

plt.style.use('ggplot')

# Fix the seed, so outputs are exactly reproducible
torch.manual_seed(12345);

# Use the GPU if available
def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")
device = detect_device()

## 3.1 FashionMNIST again

We will work with the same code as last week, with some slight modifications.
So, we will again do experiments with the [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist).

In [ ]:
fashionmnist = torchvision.datasets.FashionMNIST(
    root=".", download=True,
    transform=torchvision.transforms.Compose([
         torchvision.transforms.ToTensor(),
         lambda img: img.flatten()
    ]))

# use 1000 samples for training, 500 for validation, ignore the rest
fashion_train, fashion_validation, _ = torch.utils.data.random_split(
    fashionmnist, [1000, 500, len(fashionmnist) - (1000 + 500)])

The neural network is the same as in assignment 2.
We include our implementation of `build_net` here. You may substitute your own implementation instead, if you prefer.

In [ ]:
def build_net():
    return torch.nn.Sequential(
        torch.nn.Linear(784, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 10)
    )

The training code is also almost the same as last time, except that we add an extra parameter to `fit`.

In [ ]:
def fit(net, train, validation, optimizer, epochs=25, batch_size=10, after_epoch=None, device=device):
    """
    Train and evaluate a network.
     - net:               the network to optimize
     - train, validation: the training and validation sets
     - optimizer:         the optimizer (such as torch.optim.SGD())
     - epochs:            the number of epochs to train
     - batch_size:        the batch size
     - after_epoch:       optional function to call after every epoch
     - device:            whether to use a gpu ('cuda') or the cpu ('cpu')

    Returns a dictionary of training and validation statistics.
    """

    # move the network parameters to the gpu, if necessary
    net = net.to(device)

    # initialize the loss and accuracy history
    history = collections.defaultdict(list)
    epoch_stats, phase = None, None

    # initialize the data loaders
    data_loader = {
        'train':      torch.utils.data.DataLoader(train, batch_size=batch_size, shuffle=True),
        'validation': torch.utils.data.DataLoader(validation, batch_size=batch_size)
    }

    # measure the length of the experiment
    start_time = time.time()

    # some advanced PyTorch to look inside the network and log the outputs
    # you don't normally need this, but we use it here for our analysis
    def register_measure_hook(idx, module):
        def hook(module, input, output):
            with torch.no_grad():
                # store the mean output values
                epoch_stats['%s %d: %s output mean' % (phase, idx, type(module).__name__)] += \
                    output.mean().detach().cpu().numpy()
                # store the mean absolute output values
                epoch_stats['%s %d: %s output abs mean' % (phase, idx, type(module).__name__)] += \
                    output.abs().mean().detach().cpu().numpy()
                # store the std of the output values
                epoch_stats['%s %d: %s output std' % (phase, idx, type(module).__name__)] += \
                    output.std().detach().cpu().numpy()
        module.register_forward_hook(hook)

    # store the output for all layers in the network
    for layer_idx, layer in enumerate(net):
        register_measure_hook(layer_idx, layer)
    # end of the advanced PyTorch code

    for epoch in tqdm.tqdm(range(epochs), desc='Epoch', leave=False):
        # initialize the loss and accuracy for this epoch
        epoch_stats = collections.defaultdict(float)
        epoch_stats['train steps'] = 0
        epoch_stats['validation steps'] = 0
        epoch_outputs = {'train': [], 'validation': []}

        # first train on training data, then evaluate on the validation data
        for phase in ('train', 'validation'):
            # switch between train and validation settings
            net.train(phase == 'train')

            epoch_steps = 0
            epoch_loss = 0
            epoch_accuracy = 0

            # loop over all minibatches
            for x, y in data_loader[phase]:
                # move data to gpu, if necessary
                x = x.to(device)
                y = y.to(device)

                # compute the forward pass through the network
                pred_y = net(x)

                # compute the current loss and accuracy
                loss = torch.nn.functional.cross_entropy(pred_y, y)
                pred_class = torch.argmax(pred_y, dim=1)
                accuracy = torch.mean((pred_class == y).float())

                # add to epoch loss and accuracy
                epoch_stats[f'{phase} loss'] += loss.detach().cpu().item()
                epoch_stats[f'{phase} accuracy'] += accuracy.detach().cpu().item()

                # store outputs for later analysis
                epoch_outputs[phase].append(pred_y.detach().cpu().numpy())

                # only update the network in the training phase
                if phase == 'train':
                    # set gradients to zero
                    optimizer.zero_grad()

                    # backpropagate the gradient through the network
                    loss.backward()

                    # track the gradient and weight of the first layer
                    # (not standard; we only need this for the assignment)
                    epoch_stats['train mean abs grad'] += \
                        torch.mean(torch.abs(net[0].weight.grad)).detach().cpu().item()
                    epoch_stats['train mean abs weight'] += \
                        torch.mean(torch.abs(net[0].weight)).detach().cpu().item()

                    # update the weights
                    optimizer.step()

                epoch_stats[f'{phase} steps'] += 1

            # compute the mean loss and accuracy over all minibatches
            for key in epoch_stats:
                if phase in key and not 'steps' in key:
                    epoch_stats[key] = epoch_stats[key] / epoch_stats[f'{phase} steps']
                    history[key].append(epoch_stats[key])

            # count the number of update steps
            history[f'{phase} steps'].append((epoch + 1) * epoch_stats[f'{phase} steps'])

            # store the outputs
            history[f'{phase} outputs'].append(np.concatenate(epoch_outputs[phase]).flatten())

        history['epochs'].append(epoch)
        history['time'].append(time.time() - start_time)

        # call the after_epoch function
        if after_epoch is not None:
            stop = after_epoch(net, epoch, epoch_stats)
            if stop is Stop:
                break

    return history

# marker to indicate stopping
Stop = "stop"

In [ ]:
# helper code to plot our results
class HistoryPlotter:
    def __init__(self, plots, table, rows, cols, param_names=[]):
        self.plots = plots
        self.table = table
        self.rows = rows
        self.cols = cols
        self.histories = {}
        self.results = []
        self.params = []
        self.param_names = []

        self.fig, self.axs = plt.subplots(ncols=cols * len(plots), nrows=rows,
                                          sharex='col', sharey='col',
                                          figsize=(3.5 * cols * len(plots), 3 * rows))
        plt.tight_layout()
        IPython.display.display(self.fig)
        IPython.display.clear_output(wait=True)

    # add the results of an experiment to the plot
    def add(self, title, history, row, col, epoch=-1, param=None):
        self.histories[title] = history
        self.results.append((title, {key: history[key][epoch] for key in self.table}))
        self.params.append(param)

        for plot_idx, plot_xy in enumerate(self.plots):
            ax = self.axs[row, col * len(self.plots) + plot_idx]
            for key in plot_xy['y']:
                lines = ax.plot(history[plot_xy['x']], history[key], label=key)
                if epoch >= 0:
                    ax.plot([history[plot_xy['x']][epoch]], [history[key][epoch]], marker='*', color='black')
            if 'accuracy' in plot_xy['y'][0]:
                ax.set_ylim([0, 1.01])
            ax.legend()
            ax.set_xlabel(plot_xy['x'])
            ax.set_title(title)
        plt.tight_layout()
        IPython.display.clear_output(wait=True)
        IPython.display.display(self.fig)

    # print a table of the results for all experiments
    def print_table(self):
        df = pd.DataFrame([
            { 'experiment': title, **{key: row[key] for key in self.table} }
            for title, row in self.results
        ])
        IPython.display.display(df)

    def done(self):
        plt.close()
        self.print_table()

## 3.2 Weight decay (6 points)

The training can be regularized using weight decay. This option is built-in in many of the PyTorch optimizers [(documentation)](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#torch.optim.Adam).

We will set up an experiment to investigate how this affects the training of the model.

We use the good settings from last week:
* Optimizer: Adam
* Learning rate: 0.0001
* Minibatch size: 32
* 150 epochs

and apply L2 weight decay with a factor 0, 0.0001, 0.001, 0.01, or 0.1.

**(a) Complete the code below and run the experiment. <span style="float:right"> (1 point)</span>**

In [ ]:
plotter_weight_decay = \
    HistoryPlotter(plots=[{'x': 'epochs', 'y': ['train loss', 'validation loss']},
                          {'x': 'epochs', 'y': ['train accuracy', 'validation accuracy']},
                          {'x': 'epochs', 'y': ['train mean abs weight']},],
                   table=['train loss', 'validation loss', 'train accuracy', 'validation accuracy', 'train mean abs weight'],
                   param_names=['weight_decay'],
                   rows=5, cols=1)

epochs = 150
lr = 0.0001
batch_size = 32
weight_decays = [0, 0.0001, 0.001, 0.01, 0.1]

for row, weight_decay in enumerate(weight_decays):
    net = build_net()
    # TODO: Set up optimizer with the given weight_decay

    optimizer = torch.optim.Adam(net.parameters(), weight_decay=weight_decay, lr=lr)

    history_weight_decay = fit(net, fashion_train, fashion_validation, optimizer=optimizer, epochs=epochs, batch_size=batch_size)
    plotter_weight_decay.add(f'weight_decay={weight_decay}', history_weight_decay, row=row, col=0, param=weight_decay)

plotter_weight_decay.done()

**(b) How can you observe the amount of overfitting in the plots? <span style="float:right"> (1 point)</span>**

ANSWER:  
You can clearly observe the amount of overfitting, by comparing the training and validation loss in the left most panel. With overfitting on the training set, you a greater divide between the loss curves, which means that the model performs well on the training set (high training accuracy), but not so great on the validation set.
Upon introducing higher levels of weight decay, the training accuracy and validation accuracy curves moves closer together, but also the loss is still higher.
Moreover, the overfitting is also mirrored in the behavior of the train mean abs weight, which changes direction (drops), after a certain weight decay has been introduced (transition point: 0.01).


**SHORT ANSWER:**
Overfitting is evident when there is a difference between training & validation loss. The two lines (train and validation) get closer together when there is less overfitting.

**(c) How does weight decay affect the performance of the model in the above experiments? Give an explanation in terms of the amount of overfitting. <span style="float:right"> (1 point)</span>**

**ANSWER:**  
With increasing weight decay, the training loss increases drastically (I guess its not a linear relationship). For example there is quite a large jump in training loss from weight_decay = 0.01 and weight_decay = 0.1.

However, weight decay not only significantly reduces training accuracy, but can also reduce validation accuracy to quite a high degree (see validation accuracy for decay of 0.1).

**SOLUTION:**  
More weight decay reduces overfitting, i.e., it decreases the difference between train and validation accuracy. But for high amounts of weight decay, this comes at the cost of decreasing both training & validation accuracy. In that case the model is underfitting.

In these experiments you have implemented weight decay using the built in weight decay feature of the optimizer.

An alternative is to add an L2 penaltiy to the loss:
\begin{align*}
L_\text{regularized}(\theta) = L(\theta) + \lambda \frac{1}{2} |\theta|_2^2,
\end{align*}
which results in a gradient
\begin{align*}
\nabla_\theta L_\text{regularized}(\theta) = \nabla_\theta L(\theta) + \lambda \theta.
\end{align*}
Using gradient descend will then decrease the weights.

**(d) Are the two ways of implementing weight decay equivalent when using the Adam optimizer? <span style="float:right"> (1 point)</span>**

Hint: look at [the torch documentation for Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html).

**ANSWER:**  
Yes, these two ways are equivalent. By default ADAM uses an L2 penalty.

However, there is one problem wth regular ADAM:
"Adding the L2 regularization term to the loss affects the adaptive learning rates, which can hinder optimal convergence".


Another way to implement weight decay, and where the name comes from, is to scale or decay the weights directly:
\begin{align*}
\theta \gets (1 - \lambda) \theta
\end{align*}

Combined with Adam, this gives the [`AdamW` optimizer](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html).

**(e) Which of the following statements about Adam and AdamW are true? <span style="float:right"> (1 point)</span>**

[a] They are equivalent algorithms.  
[b] In Adam, the weight decay is added to the loss function itself.  
[c] AdamW applies momentum and gradient scaling to the weight decay.   
[d] Adam decouples weight decay from adapting the learning rate.  

AdamW is a smarter version of Adam, as it decouples weight decay from the gradient update step.

Instead of adding weight decay to the loss function, it applies weight decay directly during the parameter update, leading to more consistent regularization and better generalization.

TODO: Select one answer:  
[b]

**ELABORATED:**  
The two algorithms are very similar, but Adam applies momentum and gradient scaling to the weight decay term as well, whereas AdamW doesnot.
This way, AdamW (not Adam) decouples the weight decay from calculating the adaptive learning rate.

### Learning curves

So far the only learning curves we have looked at have the number of epochs on the horizontal axis. We can also make learning curves putting another parameter on that axis.

**(f) Run the code below to plot a learning curve <span style="float:right"> (no points)</span>**

In [ ]:
weight_decays = torch.tensor(plotter_weight_decay.params) + 1e-7
  # Note: add 1e-7 to prevent log(0)

fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(4.5 * 2, 3))
for i, stat in enumerate(['loss', 'accuracy']):
    keys = [f'train {stat}', f'validation {stat}']
    values = {key: [r[1][key][-1] for r in plotter_weight_decay.histories.items()] for key in keys}

    ax = axs[i]
    for key in keys:
        ax.plot(weight_decays, values[key], '.-', label=key)
    ax.set_xscale('log');
    ax.set_xlabel('weight decay');
    ax.set_ylabel(stat);
    ax.legend();

**(g) Looking a this learning curve, how do you see signs of overfitting or underfitting? <span style="float:right"> (1 point)</span>**

**ANSWER:**
Plotting the training and validation loss as a function of weight decay shows that while there is still a large divide / distance between the the two curves + the training loss being very low is a sign of overfitting.
On the other hand, we can observe underfitting when both the training and validation loss shoot up and mirrored by that: the training accuracy and validation accuracy drop significantly (this happens at weight decay of 0.1).
The sweetspot, where we can see a good balance between overfitting and underfitting happens when we choose a weight decay of 0.01.

## 3.3 Dropout (9 points)

Next, we will do experiments with dropout. This gives another way of regularizing the training.

**(a) Make a copy of the network architecture below, and add dropout.<span style="float:right"> (1 point)</span>**

Add dropout layers after each linear layer, except for the last.

Hint: see [torch.nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html).

In [ ]:
def build_net_with_dropout(p):
    # TODO: network with dropout layers, with dropout probability p
    return torch.nn.Sequential(
        torch.nn.Linear(784, 128),
        # add dropout layer - during training, randomly zeroes some elements of
        # the input tensor with probablity p (default = 0.5)
        torch.nn.Dropout(p = p),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 64),
        torch.nn.Dropout(p = p),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 10)
    )

**(b) Should you put dropout layers before or after ReLU activation functions? Does it matter?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Intuitively I would think that you should put dropout layers before the activation function.

But a quick google search shows that you usually place a dropout layer after an activation function.

But I am not sure if, how and why it matters?
It doesn't really matter,

**CORRECT ANSWER:**  
Conceptually, when you apply dropout directly after the linear layeryou are dropping the outputs of that layer. When you apply dropout after the activation function, dropout is done before the next layer, so you are dropping inputs of the next layer.The latter is how dropout is usually described.

Here it doesn't matter, because `dropout(relu(x)) = relu(dropout(x))`, both are equal to `max(0, x)` with probability (1-p) and to 0 with probability p.

**(c) How would the model be affacted if you put a dropout layer after the last linear layer?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
It doesn't make sense to include a dropout layer after the last linear layer, because this is our output layer which computes the final classification, and we don't want our final prediction be affected by dropout.

**CORRECT ANSWER**:  
Then with probability $p$ the prediction for the true class would be 0. There would also be no gradient propagated through the dropout layer in that case, so that training sample is wasted. This would slow down training for no bennefit.


**(d) Set up an experiment to see how dropout affects our results. <span style="float:right"> (1 point)</span>**

In [ ]:
plotter_dropout = \
    HistoryPlotter(plots=[{'x': 'epochs', 'y': ['train loss', 'validation loss']},
                          {'x': 'epochs', 'y': ['train accuracy', 'validation accuracy']},],
                   table=['train loss', 'validation loss', 'train accuracy', 'validation accuracy', 'train mean abs weight'],
                   param_names=['dropout'],
                   rows=3, cols=2)

epochs = 150
lr = 0.0001
batch_size = 32
dropouts = [0, 0.1, 0.2, 0.3, 0.6, 0.9]

for row, dropout in enumerate(dropouts):
    # TODO: Set up a network with the right dropout, and an optimizer

    net = build_net_with_dropout(p=dropout)

    optimizer = torch.optim.Adam(net.parameters(), weight_decay=weight_decay, lr=lr)

    history_dropout = fit(net, fashion_train, fashion_validation, optimizer=optimizer, epochs=epochs, batch_size=batch_size)
    plotter_dropout.add(f'dropout={dropout}', history_dropout, row=row//2, col=row%2, param=dropout)

plotter_dropout.done()

**(e) How does dropout affect the results?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
In general, introducing dropout leads to a strong reduction in accuracy (both for training and validation set).

Elaborate a bit more.

**(f) How does dropout affect training speed? Has the training converged in all runs?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
With higher dropout, the training speed is slower / training converges more slowly.
Training has not converged in all runs, for example the loss still seems to decrease with a dropout of 0.9.

Double check & elaborate a bit more.

**(g) With a large amount of dropout, the training loss can be worse than the validation loss. How is this possible?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Because dropout is only applied during training, where it adds noise and this noise is not present during validation/testing, so it makes sense that the training loss can be worse than he validation loss. Especially when you introduce a large amount of dropout you increase the dissimilarity between the training run and the validation run.

**CORRECT SOLUTION**:
The train loss is computed with dropout, the validation loss without dropout. With a large dropout rate this adds a lot of noise, leading to worse results.

**(h) Plot learning curves for the dropout parameter (train+validation loss, train+validation accuracy).<span style="float:right"> (1 point)</span>**

You should use a linear scale for the x-axis (dropout parameter). Be sure to use the right histories, and to label your axes.

In [ ]:
# TODO: Plot a learning curve
dropouts = torch.tensor(plotter_dropout.params) + 1e-7
  # Note: add 1e-7 to prevent log(0)

fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(4.5 * 2, 3))
for i, stat in enumerate(['loss', 'accuracy']):
    keys = [f'train {stat}', f'validation {stat}']
    values = {key: [r[1][key][-1] for r in plotter_dropout.histories.items()] for key in keys}

    ax = axs[i]
    for key in keys:
        ax.plot(dropouts, values[key], '.-', label=key)
    ax.set_xscale('linear');
    ax.set_xlabel('dropout');
    ax.set_ylabel(stat);
    ax.legend();


**(i) In the above learning curve, where can we find the optimum for the validation loss, if any?<span style="float:right"> (1 point)</span>**

**ANSWER:**   
The lowest value (i.e. the optimum) for the validation loss can we found with either None or very small amounts of dropout (up and including 0.2).

Elaborate a bit more - cause this is not really an optimum, is it?

## 3.4 Early stopping (7 points)

If you look at the learning curves of the unregularised models, you can see that the validation loss starts to go up after a certain amount of training.
It would be good to stop at that point, which is called early stopping.

There are two ways to implement early stopping:
1. Run the training for a fixed number of epochs, but keep track of the best result on the validation set.
2. Run until the validation loss does not decrease for a certain number of epochs.

Only the second option is actually early *stopping*, but the first option can be easier to implement.

**(a) Implement the first style of early stopping, and run the experiment below. <span style="float:right"> (2 points)</span>**

You can pass a function to the `after_epoch` parameter of `fit`. This function is called after every epoch.

The `epoch` parameter to `plotter.add` highlights a specific epoch in the results with a star, and selects it for the table.

In [ ]:
plotter_early_stop = \
    HistoryPlotter(plots=[{'x': 'epochs', 'y': ['train loss', 'validation loss']},
                          {'x': 'epochs', 'y': ['train accuracy', 'validation accuracy']}],
                   table=['train loss', 'validation loss', 'train accuracy', 'validation accuracy', 'epochs'],
                   rows=4, cols=1)

epochs = 150
lrs = [0.1, 0.01, 0.001, 0.0001]
batch_size = 32

for row, lr in enumerate(lrs):
    # the best network, epoch at which we found it, and stats
    best_net = None
    best_epoch = 0
    best_stats = {'train loss': torch.inf, 'validation loss': torch.inf, 'train accuracy': 0, 'validation accuracy': 0}

    def track_best(net, epoch, epoch_stats):
        global best_net, best_epoch, best_stats
        # TODO: keep track of the best model

        # something like this
        if epoch_stats['validation loss'] < best_stats['validation loss']:
          best_stats = {
                'train loss': epoch_stats['train loss'],
                'validation loss': epoch_stats['validation loss'],
                'train accuracy': epoch_stats['train accuracy'],
                'validation accuracy': epoch_stats['validation accuracy'],
                'epochs': epoch
            }
          best_net = net
          best_epoch = epoch

    net = build_net()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    history_early_stop = fit(net, fashion_train, fashion_validation, optimizer=optimizer, epochs=epochs, batch_size=batch_size, after_epoch=track_best)
    plotter_early_stop.add(f'lr={lr}', history_early_stop, row=row, col=0, epoch=best_epoch)

plotter_early_stop.done()

**(b) Looking at the results, does early stopping prevent overfitting?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Yes, in some cases early stopping does prevent overfitting on the training data. However, it also seems to interact with the learning rate.
But for some experiments, it has stopped too early, not allowing the training to converge fully.

**(c) With the first style of early stopping, you pick the network with the best validation loss. Does this mean it will always performed better in terms of generalization compared to the network trained without early stopping? Briefly explain your answer.<span style="float:right"> (1 point)</span>**

**ANSWER:**  
No, not necessarily. Because the consequences of early stopping can also be that the training hasn't fully converged yet - such that higher accuracy might be possible.

**CORRECT SOLUTION**:  
No. Evenif the network selected using early stopping has the lowest validation loss, it doesn't necessarily mean it has better generalization performance. The issue is similar to what happens during hyperparameter optimization, where we run the risk of overfitting the hyperparameters to the validation set. For a fair comparison of the two networks, you would need to evaluate their performance on an independent test set.

Copying a neural network with `net2 = net1` makes a shallow copy, that is, the two variables refer to the same network in memory.

**(d) If you used `best_net = net` in `track_best`, would `best_net` contain the optimal early stopping parameters after training for 150 epochs? If not, how could you get access to them?<span style="float:right"> (1 point)</span>**

**ANSWER:**  
Not sure here - I need to look into memory mapping.

A shallow copy is a new object created by copying the original object's top-level properties.
A shallow copy creates a new object but retains references to the objects contained within the original. It only copies the top-level structure without duplicating nested elements.


No, I would need to make a deep copy. Because otherwise the optimal early stopping parameters would be overwritten, and keep updating.

You could get access to them by creating a deep instead of a shallow copy.

### Actual early *stopping*

It is wasteful to keep training if we know that the loss is only getting worse. So we might as well stop at that point.

**(e) Implement the second variant of early stopping: stop training if the best validation loss does not decrease for 5 epochs.<span style="float:right"> (1 point)</span>**

Hint: The `fit` function will stop the training if the `after_epoch` function returns `Stop`.

In [ ]:
plotter_early_stop2 = \
    HistoryPlotter(plots=[{'x': 'epochs', 'y': ['train loss', 'validation loss']},
                          {'x': 'epochs', 'y': ['train accuracy', 'validation accuracy']}],
                   table=['train loss', 'validation loss', 'train accuracy', 'validation accuracy', 'epochs'],
                   rows=4, cols=1)

epochs = 150
lrs = [0.1, 0.01, 0.001, 0.0001]
batch_size = 32

for row, lr in enumerate(lrs):
    best_net = None
    best_epoch = 0
    best_stats = {'train loss': torch.inf, 'validation loss': torch.inf, 'train accuracy': 0, 'validation accuracy': 0}
    stop_criterion = 5
    def stop_if_no_loss_decrease(net, epoch, epoch_stats):
        global best_net, best_epoch, best_stats, stop_counter
        # TODO: return Stop if the loss does not go down for 5 epochs - yeah, okay this won't work

        # I should look into try and except statements

        current_loss = epoch_stats['validation loss']

        # If the loss improves → reset counter & save state
        if current_loss < best_stats['validation loss']:
            best_stats = epoch_stats.copy()
            best_epoch = epoch
            best_net = copy.deepcopy(net)
            stop_counter = 0
            return

        # Otherwise → no improvement
        stop_counter += 1

        # stop if validation loss hasn't decreased for 5 epochs
        if stop_counter >= stop_criterion:
            return Stop

    net = build_net()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    history_early_stop2 = fit(net, fashion_train, fashion_validation, optimizer=optimizer, epochs=epochs, batch_size=batch_size, after_epoch=stop_if_no_loss_decrease)
    plotter_early_stop2.add(f'lr={lr}', history_early_stop2, row=row, col=0)

plotter_early_stop2.done()

In [ ]:
## this is the correct implementation as specified in the solution
plotter_early_stop2 = \
    HistoryPlotter(plots=[{'x': 'epochs', 'y': ['train loss', 'validation loss']},
                          {'x': 'epochs', 'y': ['train accuracy', 'validation accuracy']}],
                   table=['train loss', 'validation loss', 'train accuracy', 'validation accuracy', 'epochs'],
                   rows=4, cols=1)

epochs = 150
lrs = [0.1, 0.01, 0.001, 0.0001]
batch_size = 32

for row, lr in enumerate(lrs):
    best_net = None
    best_epoch = 0
    best_stats = {'train loss': torch.inf, 'validation loss': torch.inf, 'train accuracy': 0, 'validation accuracy': 0}

    def stop_if_no_loss_decrease(net, epoch, epoch_stats):
        global best_net, best_epoch, best_stats
        ### BEGIN ANSWER
        if epoch_stats['validation loss'] < best_stats['validation loss']:
            best_net = net
            best_epoch = epoch
            best_stats = epoch_stats
        if epoch >= best_epoch + 5:
            return Stop
        ### END ANSWER

    net = build_net()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    history_early_stop2 = fit(net, fashion_train, fashion_validation, optimizer=optimizer, epochs=epochs, batch_size=batch_size, after_epoch=stop_if_no_loss_decrease)
    plotter_early_stop2.add(f'lr={lr}', history_early_stop2, row=row, col=0)

plotter_early_stop2.done()

**(f) With the second variant of early stopping, is the final network after training the optimal one in terms of validation loss? <span style="float:right"> (1 point)</span>**

**ANSWER:**  
Yes, but again it depends on the learning rate. For learning rates lr=0.001 and 0.0001 we get quite good results in terms of the validation loss and validation accuracy.

**CORRECT SOLUTION**:  
No, we only stop if the validation loss did not decrease for 5 epochs, so the network has gotten a bit worse during that time.

## 3.5 Hyperparameter optimization (3 points)

We have seen quite a few hyperparameters this week and last week.
To pick the optimal parameters, one strategy would be to do what we have done, and run an experiment for each parameter individually.

**(a) Look at the previous experiments, and pick the best hyperparameter values.<span style="float:right"> (1 point)</span>**

Optionally: look at the experiments from assignment 2 and also pick the optimal network width.

Batch size:    batch sizes larger or equal than 32  
Optimizer:     ADAM or ADAM optimizer with regularization  
Learning rate: 0.01 or 0.001  
Dropout:       0.1 & 0.2     
Weight decay:  0.01   

**(b) If you select the hyperparameters this way, will you get the best results? Explain your answer.<span style="float:right"> (1 point)</span>**

**ANSWER:**  
No, you probably won't get the best results, as this is neither a very systematic nor exhaustive process.
To better explore the hyperparameter space, you could use grid search (there are different variants of grid search, so double check).

An alternative is to use a grid search, and try all possible combinations of hyperparameters.

**(c) How many experiments would you need to do to explore all combinations of learning rate, weight decay, and dropout that we used in this assignment?<span style="float:right"> (1 point)</span>**

nr of learning rates: 4
nr of weight decay: 4
nr of dropout: 6

So 4 x 4 x 6 = 96 experiments

So we do see that things scale up / explode relatively fast.

## The end

Well done! Please double check the instructions at the top before you submit your results.

*This assignment has 25 points.*
<span style="float:right;color:#aaa;font-size:10px;"> Version 9dde262 / 2025-11-21</span>